# Task 2 — Trends and the spaghetti problem

**Week 1 data visualization assignment**

This notebook follows the task brief and chart-design rules: titles state findings, bars use a zero baseline, color carries meaning, excluded rows are stated, and each chart has a short rationale.

## Setup and data

Run the cells from top to bottom. The CSV is read from its public source URL; an internet connection is required. If needed, install `pandas`, `numpy`, and `matplotlib` in your notebook environment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
DATA_URL = 'https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv'
df = pd.read_csv(DATA_URL)
df.head()

## Check the panel and inspect GDP per person

The panel should contain 142 countries observed in 12 five-year steps. GDP per capita is highly right-skewed, so compare its ordinary and logarithmic displays.

In [ ]:
print('Rows:', len(df), '| Countries:', df.country.nunique(), '| Years:', df.year.nunique())
print('Missing values:', int(df.isna().sum().sum()))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df.gdpPercap, bins=35, color='#6599C8', edgecolor='white'); axes[0].set_title('GDP per capita is strongly right-skewed'); axes[0].set_xlabel('GDP per capita')
axes[1].hist(df.gdpPercap, bins=35, color='#6599C8', edgecolor='white'); axes[1].set_xscale('log'); axes[1].set_title('Log x scale reveals more of the distribution'); axes[1].set_xlabel('GDP per capita (log scale)')
fig.tight_layout()

The logarithmic x-axis spreads out lower values and compresses the long upper tail. It changes the visual spacing, not the underlying values. For comparisons of ratios and a highly skewed variable, the log scale is informative; label it clearly.

## The deliberately messy country chart

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
for country, g in df.groupby('country'):
    ax.plot(g.year, g.lifeExp, linewidth=.8, alpha=.75, label=country)
ax.set(title='142 country lines make individual trends hard to read', xlabel='Year', ylabel='Life expectancy (years)')
ax.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=5, ncol=2)
fig.tight_layout()

**What is wrong:** the 142-entry legend is unusable; overlapping lines obscure one another; color is not helping distinguish countries; and the chart has no single readable message. The data are complete, so no rows were excluded.

## Fix A: aggregate by continent

In [ ]:
continent = df.groupby(['continent','year'], as_index=False).lifeExp.mean()
fig, ax = plt.subplots(figsize=(9,5))
for name, g in continent.groupby('continent'):
    ax.plot(g.year, g.lifeExp, marker='o', label=name)
ax.set(title='Life expectancy rose in every continent, with different starting points', xlabel='Year', ylabel='Mean life expectancy (years)')
ax.legend(title='Continent', frameon=False)
fig.tight_layout()

**Why a line chart:** years are ordered, and connected points make change over time easy to follow. Continent averages reveal broad regional trends while reducing clutter, but hide variation among countries.

## Fix B: highlight three countries

In [ ]:
selected = ['China', 'United States', 'Afghanistan']
fig, ax = plt.subplots(figsize=(9,5))
for _, g in df.groupby('country'):
    ax.plot(g.year, g.lifeExp, color='#D5D8DC', linewidth=.8, zorder=1)
colors = {'China':'#D55E00','United States':'#0072B2','Afghanistan':'#009E73'}
for country in selected:
    g = df[df.country.eq(country)].sort_values('year')
    if g.empty: continue
    ax.plot(g.year, g.lifeExp, color=colors[country], linewidth=2.5, zorder=3)
    ax.text(g.year.iloc[-1]+1, g.lifeExp.iloc[-1], country, color=colors[country], va='center')
ax.set(title='China, the United States, and Afghanistan follow distinct paths', xlabel='Year', ylabel='Life expectancy (years)')
fig.tight_layout()

**Why this chart:** faint grey context preserves the country-level spread while the three directly labeled trajectories provide a focused comparison without a legend. The aggregate view communicates regional averages better; the highlight view retains individual country variation.

## Has the rich–poor country gap narrowed?

In [ ]:
gdp = df.groupby('year').gdpPercap.agg(poorest='min', richest='max').reset_index()
fig, ax = plt.subplots(figsize=(9,5))
ax.fill_between(gdp.year, gdp.poorest, gdp.richest, color='#8DB7D7', alpha=.35)
ax.plot(gdp.year, gdp.poorest, color='#D55E00', label='Poorest country')
ax.plot(gdp.year, gdp.richest, color='#0072B2', label='Richest country')
ax.set_yscale('log')
ax.set(title='The richest–poorest GDP per capita gap remains enormous', xlabel='Year', ylabel='GDP per capita (log scale)')
ax.legend(frameon=False)
fig.tight_layout()
(gdp.assign(ratio=gdp.richest/gdp.poorest)[['year','ratio']].round(1))

**Why this chart:** the two boundary lines show both extremes over time, and the logarithmic scale makes their very different magnitudes readable. Compare the printed richest-to-poorest ratio across years to assess whether the multiplicative gap narrowed; extremes can be sensitive to a single country, so this answers the task's literal richest-versus-poorest question.